# rejected_signals分析ノートブック

このノートブックでは、rejected_signalsテーブルのデータを用いて、reject理由・シンボル・スコア・閾値・時系列分布・頻出パターン・直近N件の可視化・集計・分析を行います。

---

## 1. 必要なライブラリのインポート

In [ ]:
# 必要なライブラリのインポート
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Jupyterのグラフ表示設定
%matplotlib inline
sns.set(style="whitegrid")

## 2. データの準備
rejected_signalsテーブルからデータを抽出し、Pandas DataFrameとして読み込みます。

In [ ]:
# データベースからrejected_signalsを抽出
DB_PATH = os.path.join(os.path.dirname(os.path.dirname(__file__)), 'database', 'runtime_state.db')
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query('SELECT * FROM rejected_signals ORDER BY timestamp DESC', conn)
conn.close()

# タイムスタンプをdatetime型に変換
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
df.head()

In [ ]:
# --- 学習データ期間の確認（最低3ヶ月） ---
min_date = df['datetime'].min()
max_date = df['datetime'].max()
period_days = (max_date - min_date).days
print(f"データ期間: {min_date:%Y-%m-%d} ～ {max_date:%Y-%m-%d}（{period_days}日間）")
if period_days < 90:
    print("警告: 学習データ期間が3ヶ月未満です。分析・学習結果の信頼性に注意してください。")
else:
    print("学習データ期間は十分です。")

## 3. 前処理の実装
欠損値処理や型変換、必要に応じて特徴量エンジニアリングを行います。

In [ ]:
# 欠損値の確認と処理
df = df.fillna({'alert_name': 'unknown', 'side': 'unknown', 'score': -1, 'threshold': -1, 'reason': 'unknown'})

# extra列はJSON文字列なので、必要に応じて辞書型に変換可能
import ast
def parse_extra(val):
    try:
        return ast.literal_eval(val) if isinstance(val, str) else {}
    except Exception:
        return {}
df['extra_dict'] = df['extra'].apply(parse_extra)

df.info()

## 4. モデルの構築
reject理由やスコア・閾値・シンボルなどから、どの要因がrejectに寄与しているかを簡易的な分類モデルで分析します。

In [ ]:
# 時系列順でtrain/test分割（過去3ヶ月以上を担保）
# データを時系列でソート
df = df.sort_values('datetime')

# 80%をtrain, 20%をtestに分割（時系列分割）
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

# 特徴量エンコーディング
le_symbol = LabelEncoder()
train_df['symbol_enc'] = le_symbol.fit_transform(train_df['symbol'])
test_df['symbol_enc'] = le_symbol.transform(test_df['symbol'])
le_reason = LabelEncoder()
train_df['reason_enc'] = le_reason.fit_transform(train_df['reason'])
test_df['reason_enc'] = le_reason.transform(test_df['reason'])

X_train = train_df[['score', 'threshold', 'symbol_enc']]
y_train = train_df['is_score_reject']
X_test = test_df[['score', 'threshold', 'symbol_enc']]
y_test = test_df['is_score_reject']

model = RandomForestClassifier(n_estimators=50, random_state=42)

## 5. 学習の実行
用意したデータでモデルの学習（fit）を実行します。

In [ ]:
# モデルの学習
model.fit(X_train, y_train)

## 6. 評価指標の計算
テストデータを使ってモデルの精度や評価指標（accuracy, confusion matrixなど）を計算します。

In [ ]:
# モデルの評価
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = model.predict(X_test)
print('Accuracy:', accuracy_score(y_test, y_pred))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))
print('Classification Report:\n', classification_report(y_test, y_pred))

In [ ]:
# 過学習チェック（train/test精度比較）
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

from sklearn.metrics import accuracy_score
train_acc = accuracy_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)
print(f"Train Accuracy: {train_acc:.3f}")
print(f"Test Accuracy:  {test_acc:.3f}")
if train_acc - test_acc > 0.1:
    print("警告: 過学習の可能性あり（trainとtestの精度差が大きい）")
else:
    print("過学習の兆候は見られません。")

## 7. 予測の実行
新しいデータやテストデータに対してモデルを使って予測を行います。

In [ ]:
# テストデータでの予測例
sample = X_test.iloc[:5]
pred = model.predict(sample)
print('予測結果:', pred)
print('実際の値:', y_test.iloc[:5].values)